# Crossref components that are part of a work (oxjob #2065, step 4)

Crossref types 9.9M DOIs `component`: figures, tables, supporting files and review letters of an article (PLOS
`.g001`, ACS `.s001`, PeerJ `/table-4`), and also whole records that are not part of anything (Oxford English
Dictionary entries, OECD tables, Protein Data Bank structures). `ingest/Crossref.py` drops them all. This notebook
lets in the ones whose article is a work, as records the Union pipeline reads through its own flow
(`locations_parsed_crossref_component`), so each becomes a work of its own that `Work_Parents` hangs under its article.

```
crossref_items (change feed) --> crossref_component_records   every component DOI, its latest record
                             --> crossref_component_works     the admitted ones, shaped like a locations_parsed row
                                   --> Union pipeline flow --> locations_parsed --> ... --> a work per component
                                   --> Work_Parents (source crossref_component, relationship component)
                                   --> location matcher frame (left out: these mint on their DOI)
```

**Admitted** when all hold: Crossref's type is `component`; the DOI is not a work already; the DOI minus its last
`.` `/` or `-` segment (up to three times, or minus a glued `ds1`/`sup1` ending) is the DOI of a live work, first such
stem wins; that work is not typed paratext, report or dataset; the prefix is not 10.1787 (OECD). The suffix alone
never makes a component: it matches 25,612 real works.

**What the record says.** Components are mostly blank (PLOS: no title, authors or license), so the title is made from
the suffix and the parent ("Figure 2 of <parent title>"), unless the record has a real one (not a file name, not a
copy of the parent's). License, language and the open flag are the parent's. No title key (`normalized_title` NULL), so nothing
ever matches or merges on the made-up title, and no authors.

Runs as its own job (`jobs/crossref_components.yaml`) before the nightly; the nightly reads what is there. A parent
that arrives tonight gets its components tomorrow. Revert: pause the job and
`DELETE FROM openalex.crossref.crossref_component_works` (the flow passes the deletes on; the works leave the next night).

In [0]:
CREATE TABLE IF NOT EXISTS openalex.crossref.crossref_component_records (
  native_id STRING COMMENT 'the component DOI, lowercase',
  prefix STRING COMMENT 'the owner prefix Crossref reports; can differ from the prefix in the DOI', publisher STRING, raw_type STRING, title STRING, n_authors INT,
  created_date DATE, published_date DATE, updated_date DATE COMMENT 'Crossref indexed date, as crossref_exploded.updated_date',
  indexed_ts BIGINT, url STRING, ingested_at TIMESTAMP,
  items_version BIGINT COMMENT 'crossref_items version this record was read at'
)
CLUSTER BY (native_id)
COMMENT 'oxjob #2065: every Crossref DOI typed component, latest record; built by notebooks/ingest/CreateCrossrefComponentWorks'

In [0]:
CREATE TABLE IF NOT EXISTS openalex.crossref.crossref_component_state (items_version BIGINT, records_read BIGINT, run_ts TIMESTAMP)
COMMENT 'oxjob #2065: the crossref_items version CreateCrossrefComponentWorks has read up to; one row per run'

In [0]:
-- Shaped like a locations_parsed row (the Union flow selects those columns by name), plus how it was admitted.
CREATE TABLE IF NOT EXISTS openalex.crossref.crossref_component_works
CLUSTER BY (native_id)
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true', 'delta.enableDeletionVectors' = 'true')
COMMENT 'oxjob #2065: Crossref components admitted as works (their parent is a work); read by the Union pipeline flow locations_parsed_crossref_component, Work_Parents and the location matcher frame'
AS SELECT * EXCEPT (key_provenance),
  CAST(NULL AS STRING) AS parent_doi, CAST(NULL AS BIGINT) AS parent_work_id, CAST(NULL AS STRING) AS parent_stem,
  CAST(NULL AS STRING) AS component_kind, CAST(NULL AS STRING) AS title_source, CAST(NULL AS TIMESTAMP) AS admitted_at
FROM openalex.works.locations_parsed WHERE 1 = 0

### New component records

Reads `crossref_items` through its change feed from the last version read (`crossref_component_state`), so a night costs
the night's files, not the 8.5 TB table. The first run (no state row) reads the whole table once: that is the backfill.
The feed keeps about 30 days; if the job was off longer than that the feed read fails, and the fix is to delete the
state rows and let one run read the whole table again (every step here is idempotent: a record read twice is one row).

In [0]:
BEGIN
  DECLARE to_v BIGINT;
  DECLARE from_v BIGINT;
  SET to_v = (SELECT max(version) FROM (DESCRIBE HISTORY openalex.crossref.crossref_items));
  SET from_v = (SELECT max(items_version) + 1 FROM openalex.crossref.crossref_component_state);
  IF from_v IS NULL THEN
    MERGE INTO openalex.crossref.crossref_component_records t USING (WITH x AS (SELECT explode(items) AS it, ingested_at FROM openalex.crossref.crossref_items),
c AS (
  SELECT lower(trim(it.DOI)) AS native_id, it.prefix AS prefix, it.publisher AS publisher, it.type AS raw_type,
    nullif(substr(trim(get(it.title, 0)), 1, 5000), '') AS title, CAST(coalesce(size(it.author), 0) AS INT) AS n_authors,
    to_date(it.created.`date-time`) AS created_date,
    -- as ingest/Crossref.py: the earliest of issued, created, published, deposited
    least(try_to_date(format_string('%04d-%02d-%02d', get(get(it.issued.`date-parts`, 0), 0), coalesce(get(get(it.issued.`date-parts`, 0), 1), 1), coalesce(get(get(it.issued.`date-parts`, 0), 2), 1))),
          to_date(it.created.`date-time`),
          try_to_date(format_string('%04d-%02d-%02d', get(get(it.published.`date-parts`, 0), 0), coalesce(get(get(it.published.`date-parts`, 0), 1), 1), coalesce(get(get(it.published.`date-parts`, 0), 2), 1))),
          to_date(it.deposited.`date-time`)) AS published_date,
    try_to_date(format_string('%04d-%02d-%02d', get(get(it.indexed.`date-parts`, 0), 0), coalesce(get(get(it.indexed.`date-parts`, 0), 1), 1), coalesce(get(get(it.indexed.`date-parts`, 0), 2), 1))) AS updated_date,
    it.indexed.timestamp AS indexed_ts, coalesce(it.resource.primary.URL, it.URL) AS url, ingested_at
  FROM x WHERE it.type = 'component' AND it.DOI IS NOT NULL
)
SELECT c.*, CAST(to_v AS BIGINT) AS items_version FROM c
QUALIFY row_number() OVER (PARTITION BY native_id ORDER BY indexed_ts DESC NULLS LAST, ingested_at DESC NULLS LAST) = 1) s ON t.native_id = s.native_id
    WHEN MATCHED AND coalesce(s.indexed_ts, 0) >= coalesce(t.indexed_ts, 0) THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *;
  ELSEIF from_v <= to_v THEN
    MERGE INTO openalex.crossref.crossref_component_records t USING (WITH x AS (SELECT explode(items) AS it, ingested_at FROM table_changes('openalex.crossref.crossref_items', from_v, to_v)),
c AS (
  SELECT lower(trim(it.DOI)) AS native_id, it.prefix AS prefix, it.publisher AS publisher, it.type AS raw_type,
    nullif(substr(trim(get(it.title, 0)), 1, 5000), '') AS title, CAST(coalesce(size(it.author), 0) AS INT) AS n_authors,
    to_date(it.created.`date-time`) AS created_date,
    -- as ingest/Crossref.py: the earliest of issued, created, published, deposited
    least(try_to_date(format_string('%04d-%02d-%02d', get(get(it.issued.`date-parts`, 0), 0), coalesce(get(get(it.issued.`date-parts`, 0), 1), 1), coalesce(get(get(it.issued.`date-parts`, 0), 2), 1))),
          to_date(it.created.`date-time`),
          try_to_date(format_string('%04d-%02d-%02d', get(get(it.published.`date-parts`, 0), 0), coalesce(get(get(it.published.`date-parts`, 0), 1), 1), coalesce(get(get(it.published.`date-parts`, 0), 2), 1))),
          to_date(it.deposited.`date-time`)) AS published_date,
    try_to_date(format_string('%04d-%02d-%02d', get(get(it.indexed.`date-parts`, 0), 0), coalesce(get(get(it.indexed.`date-parts`, 0), 1), 1), coalesce(get(get(it.indexed.`date-parts`, 0), 2), 1))) AS updated_date,
    it.indexed.timestamp AS indexed_ts, coalesce(it.resource.primary.URL, it.URL) AS url, ingested_at
  FROM x WHERE it.type = 'component' AND it.DOI IS NOT NULL
)
SELECT c.*, CAST(to_v AS BIGINT) AS items_version FROM c
QUALIFY row_number() OVER (PARTITION BY native_id ORDER BY indexed_ts DESC NULLS LAST, ingested_at DESC NULLS LAST) = 1) s ON t.native_id = s.native_id
    WHEN MATCHED AND coalesce(s.indexed_ts, 0) >= coalesce(t.indexed_ts, 0) THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *;
  END IF;
  IF from_v IS NULL OR from_v <= to_v THEN
    INSERT INTO openalex.crossref.crossref_component_state SELECT to_v, (SELECT count(*) FROM openalex.crossref.crossref_component_records WHERE items_version = to_v), current_timestamp();
  END IF;
END

### Admit: components whose parent is a work

`doi_works` is the same DOI -> work lookup `Work_Parents` uses (a Crossref or DataCite record pinned to a work), so
every admitted component gets its edge there. A stem that is itself an admitted component is never a parent (no
component-of-a-component chains). A component DOI that is a work already (pinned in the registry: Protein Data Bank
structures, records Crossref once typed otherwise; or the DOI of a live work: repository copies of a figure), or whose
cleaned DOI the work id map holds, is left alone: 357K on 13 Nov 2026.

In [0]:
INSERT INTO openalex.crossref.crossref_component_works BY NAME
WITH cand AS (
  SELECT r.native_id, split(r.native_id, '/')[0] AS prefix, nullif(trim(regexp_replace(regexp_replace(r.title, '</?[a-zA-Z][a-zA-Z0-9:-]*( [^<>]*)?/?>', ''), '\\s+', ' ')), '') AS rec_title, r.publisher, r.created_date, r.published_date, r.updated_date, r.ingested_at
  FROM openalex.crossref.crossref_component_records r
  LEFT ANTI JOIN openalex.crossref.crossref_component_works w ON w.native_id = r.native_id
  WHERE r.native_id NOT LIKE '10.1787/%' AND r.native_id RLIKE '^10[.][0-9]{4,}/.+'
),
stems AS (
  SELECT c.native_id,
    regexp_extract(c.native_id, '^(.*)[./-][^./-]+$', 1) AS s1,
    regexp_replace(c.native_id, '(ds|sd|sup|supp|s|m|v)[0-9]+$', '') AS s_glued,
    regexp_extract(regexp_extract(c.native_id, '^(.*)[./-][^./-]+$', 1), '^(.*)[./-][^./-]+$', 1) AS s2,
    regexp_extract(regexp_extract(regexp_extract(c.native_id, '^(.*)[./-][^./-]+$', 1), '^(.*)[./-][^./-]+$', 1), '^(.*)[./-][^./-]+$', 1) AS s3,
    length(split(c.native_id, '/')[0]) + 1 AS min_len
  FROM cand c
),
long_stems AS (  -- one row per (component, stem) in the order tried; a stem must be longer than the bare prefix
  SELECT native_id, pos, stem FROM stems
  LATERAL VIEW posexplode(array(s1, s_glued, s2, s3)) e AS pos, stem
  WHERE stem IS NOT NULL AND stem <> native_id AND length(stem) > min_len
),
doi_works AS (
  SELECT l.native_id AS doi, MAX_BY(l.work_id, l.provenance = 'crossref') AS work_id
  FROM openalex.works.location_work_ids l
  LEFT SEMI JOIN (SELECT DISTINCT stem FROM long_stems) s ON s.stem = l.native_id
  LEFT ANTI JOIN openalex.crossref.crossref_component_works w ON w.native_id = l.native_id
  WHERE l.native_id_namespace = 'doi' AND l.provenance IN ('crossref', 'datacite') AND l.work_id IS NOT NULL
  GROUP BY l.native_id
),
is_work AS (  -- the component DOI itself is pinned to a work, is the DOI of a live work (a repository copy of the
  -- figure that carries the publisher's DOI: 87K Zenodo and Figshare copies on 13 Nov 2026), or the map holds its cleaned form
  SELECT l.native_id FROM openalex.works.location_work_ids l LEFT SEMI JOIN cand c ON c.native_id = l.native_id
  WHERE l.native_id_namespace = 'doi' AND l.work_id IS NOT NULL
  UNION
  SELECT c.native_id FROM cand c
  LEFT SEMI JOIN openalex.works.openalex_works w ON lower(w.doi) = concat('https://doi.org/', c.native_id)
  UNION
  SELECT c.native_id FROM cand c
  LEFT SEMI JOIN openalex.works.work_id_map m
    ON regexp_replace(lower(m.doi), '[^a-zA-Z0-9\\./-]', '') = regexp_replace(c.native_id, '[^a-zA-Z0-9\\./-]', '')
  UNION  -- or an admitted component's DOI cleans to the same string (the map learns it only at the next nightly)
  SELECT c.native_id FROM cand c
  LEFT SEMI JOIN openalex.crossref.crossref_component_works w
    ON regexp_replace(w.native_id, '[^a-zA-Z0-9\\./-]', '') = regexp_replace(c.native_id, '[^a-zA-Z0-9\\./-]', '')
),
resolved AS (
  SELECT s.native_id, MIN_BY(s.stem, s.pos) AS parent_doi, MIN_BY(d.work_id, s.pos) AS parent_work_id,
         MIN_BY(element_at(array('strip1', 'strip_glued', 'strip2', 'strip3'), s.pos + 1), s.pos) AS parent_stem
  FROM long_stems s JOIN doi_works d ON d.doi = s.stem
  LEFT ANTI JOIN is_work iw ON iw.native_id = s.native_id
  GROUP BY s.native_id
),
one_per_clean_doi AS (  -- two component DOIs that clean to the same string would share one work: keep the first
  SELECT c.*, r.parent_doi, r.parent_work_id, r.parent_stem
  FROM cand c JOIN resolved r ON r.native_id = c.native_id
  QUALIFY row_number() OVER (PARTITION BY regexp_replace(c.native_id, '[^a-zA-Z0-9\\./-]', '') ORDER BY c.native_id) = 1
),
t AS (
SELECT r.*, p.type AS parent_type, p.title AS parent_title, p.open_access.is_oa AS parent_is_oa,
  coalesce(p.best_oa_location.license, p.primary_location.license) AS parent_license,
  p.language AS parent_language,
  k.component_kind,
  CASE WHEN k.keep_title AND k.label_only THEN 'record_label' WHEN k.keep_title THEN 'record' ELSE 'made' END AS title_source,
  substr(CASE
    WHEN k.keep_title AND k.label_only THEN concat(regexp_replace(r.rec_title, '[\\s.:;,]+$', ''), ' of ', trim(p.title))
    WHEN k.keep_title THEN r.rec_title
    ELSE concat(
      CASE k.component_kind
        WHEN 'peer_review' THEN 'Peer review history'
        WHEN 'figure' THEN concat_ws(' ', 'Figure', k.num)
        WHEN 'table' THEN concat_ws(' ', 'Table', k.num)
        WHEN 'supporting_information' THEN concat_ws(' ', 'Supporting information', k.num)
        ELSE concat_ws(' ', 'Supplementary file', k.num) END,
      ' of ', trim(p.title),
      CASE WHEN k.component_kind = 'peer_review' AND try_cast(k.num AS INT) > 1 THEN concat(' (part ', k.num, ')') ELSE '' END)
  END, 1, 5000) AS made_title
FROM (SELECT * FROM one_per_clean_doi) r
JOIN openalex.works.openalex_works p ON p.id = r.parent_work_id
LATERAL VIEW OUTER inline(array(named_struct(
  'sfx', lower(regexp_replace(substr(r.native_id, length(r.parent_doi) + 1), '^[./-]+', ''))))) s
LATERAL VIEW OUTER inline(array(named_struct(  -- letters and digits only, to compare the record's title with the parent's
  'rec', regexp_replace(lower(r.rec_title), '[^\\p{L}\\p{N}]', ''), 'par', regexp_replace(lower(p.title), '[^\\p{L}\\p{N}]', '')))) n
LATERAL VIEW OUTER inline(array(named_struct(
  'component_kind', CASE
      WHEN r.prefix = '10.1371' AND s.sfx RLIKE '^r[0-9]+$' THEN 'peer_review'
      WHEN s.sfx RLIKE '^(g|fig|figure)[-_.]?[0-9]+[a-z]?$' THEN 'figure'
      WHEN s.sfx RLIKE '^(t|tab|table)[-_.]?[0-9]+[a-z]?$' THEN 'table'
      WHEN s.sfx RLIKE '^s[0-9]+$' THEN 'supporting_information'
      ELSE 'supplementary_file' END,
  -- the number in the last segment, when it reads as a count (1-999); an internal file id gets no number
  'num', CASE WHEN try_cast(regexp_extract(s.sfx, '(?<![0-9])([0-9]{1,9})[a-z]?$', 1) AS BIGINT) BETWEEN 1 AND 999
              THEN CAST(try_cast(regexp_extract(s.sfx, '(?<![0-9])([0-9]{1,9})[a-z]?$', 1) AS BIGINT) AS STRING) END,
  -- a real title: not blank, not a file name, not the parent's title again (or nearly: ACS deposits repeat it less a Greek letter)
  'keep_title', coalesce(length(r.rec_title) >= 3
      AND r.prefix <> '10.1021'  -- ACS deposits its supporting files under a draft of the article's title, editors' queries included
      AND NOT lower(r.rec_title) RLIKE '\\.(pdf|docx?|xlsx?|pptx?|csv|tsv|txt|zip|rar|gz|7z|tiff?|jpe?g|png|gif|bmp|eps|svg|mp4|mov|avi|wmv|mpe?g|mp3|wav|cif|cml|fcf|hkl|mol|sdf|pdb|xml|json|rtf|tex|bib|mat|fasta?|nex|tre|odt|ods|html?|py|ipynb|rmd|sav|dta|kml|kmz)$'
      AND NOT (r.rec_title NOT LIKE '% %' AND r.rec_title RLIKE '[_0-9]')
      AND NOT (n.rec = n.par OR (length(n.rec) >= 10 AND n.par LIKE concat('%', n.rec, '%'))
               -- levenshtein with a cap answers -1 past the cap
               OR levenshtein(left(n.rec, 300), left(n.par, 300), 45) BETWEEN 0 AND 0.15 * least(length(n.par), 300)), false),
  -- a bare label ("Supplementary data 2", "Front Matter") says nothing alone: the parent's title is added
  'label_only', coalesce(length(r.rec_title) <= 40
      AND lower(r.rec_title) RLIKE '\\b(supplement\\w*|supporting|additional|appendix|appendices|figures?|fig|tables?|data ?sheets?|datasets?|data|videos?|movies?|audio|images?|files?|matter|presentation|multimedia|online resource)\\b', false)
))) k
WHERE p.type NOT IN ('paratext', 'report', 'dataset') AND length(trim(p.title)) > 0
)
SELECT
  'crossref' AS provenance, t.native_id, 'doi' AS native_id_namespace, t.made_title AS title,
  CAST(NULL AS STRING) AS normalized_title,
  array(named_struct('id', t.native_id, 'namespace', 'doi', 'relationship', 'self')) AS ids,
  'component' AS raw_type, 'publishedVersion' AS version, t.parent_license AS license,
  t.parent_language AS language,  -- declared, so the nightly language labeller (CreateLocationsMapped, #1871) does not read the made-up title
  t.published_date, t.created_date, t.updated_date, t.publisher,
  array(named_struct('url', concat('https://doi.org/', t.native_id), 'content_type', 'html')) AS urls,
  coalesce(t.parent_is_oa, false) AS is_oa, t.ingested_at,
  false AS authors_exist, false AS affiliations_exist,
  t.native_id AS best_doi,
  named_struct('doi', t.native_id, 'pmid', CAST(NULL AS STRING), 'arxiv', CAST(NULL AS STRING), 'title_author', CAST(NULL AS STRING)) AS merge_key,
  t.parent_doi, t.parent_work_id, t.parent_stem, t.component_kind, t.title_source, current_timestamp() AS admitted_at
FROM t

### Keep title, license and the open flag equal to the parent's

A parent that was retitled, merged, opened or relicensed since its components were admitted: the component rows follow
(the change feed carries the update to `locations_parsed`). A component whose parent is no longer a live work is left
as it is and counted below.

In [0]:
MERGE INTO openalex.crossref.crossref_component_works t
USING (
  WITH cur AS (
    SELECT w.native_id, nullif(trim(regexp_replace(regexp_replace(r.title, '</?[a-zA-Z][a-zA-Z0-9:-]*( [^<>]*)?/?>', ''), '\\s+', ' ')), '') AS rec_title, split(w.native_id, '/')[0] AS prefix, w.parent_doi, d.work_id AS parent_work_id
    FROM openalex.crossref.crossref_component_works w
    JOIN openalex.crossref.crossref_component_records r ON r.native_id = w.native_id
    JOIN (SELECT l.native_id AS doi, MAX_BY(l.work_id, l.provenance = 'crossref') AS work_id
          FROM openalex.works.location_work_ids l
          LEFT SEMI JOIN (SELECT DISTINCT parent_doi FROM openalex.crossref.crossref_component_works) p ON p.parent_doi = l.native_id
          WHERE l.native_id_namespace = 'doi' AND l.provenance IN ('crossref', 'datacite') AND l.work_id IS NOT NULL
          GROUP BY l.native_id) d ON d.doi = w.parent_doi
  )
  SELECT x.native_id, x.parent_work_id, x.made_title, x.parent_license, coalesce(x.parent_is_oa, false) AS parent_is_oa,
         x.component_kind, x.title_source
  FROM (
SELECT r.*, p.type AS parent_type, p.title AS parent_title, p.open_access.is_oa AS parent_is_oa,
  coalesce(p.best_oa_location.license, p.primary_location.license) AS parent_license,
  p.language AS parent_language,
  k.component_kind,
  CASE WHEN k.keep_title AND k.label_only THEN 'record_label' WHEN k.keep_title THEN 'record' ELSE 'made' END AS title_source,
  substr(CASE
    WHEN k.keep_title AND k.label_only THEN concat(regexp_replace(r.rec_title, '[\\s.:;,]+$', ''), ' of ', trim(p.title))
    WHEN k.keep_title THEN r.rec_title
    ELSE concat(
      CASE k.component_kind
        WHEN 'peer_review' THEN 'Peer review history'
        WHEN 'figure' THEN concat_ws(' ', 'Figure', k.num)
        WHEN 'table' THEN concat_ws(' ', 'Table', k.num)
        WHEN 'supporting_information' THEN concat_ws(' ', 'Supporting information', k.num)
        ELSE concat_ws(' ', 'Supplementary file', k.num) END,
      ' of ', trim(p.title),
      CASE WHEN k.component_kind = 'peer_review' AND try_cast(k.num AS INT) > 1 THEN concat(' (part ', k.num, ')') ELSE '' END)
  END, 1, 5000) AS made_title
FROM (SELECT * FROM cur) r
JOIN openalex.works.openalex_works p ON p.id = r.parent_work_id
LATERAL VIEW OUTER inline(array(named_struct(
  'sfx', lower(regexp_replace(substr(r.native_id, length(r.parent_doi) + 1), '^[./-]+', ''))))) s
LATERAL VIEW OUTER inline(array(named_struct(  -- letters and digits only, to compare the record's title with the parent's
  'rec', regexp_replace(lower(r.rec_title), '[^\\p{L}\\p{N}]', ''), 'par', regexp_replace(lower(p.title), '[^\\p{L}\\p{N}]', '')))) n
LATERAL VIEW OUTER inline(array(named_struct(
  'component_kind', CASE
      WHEN r.prefix = '10.1371' AND s.sfx RLIKE '^r[0-9]+$' THEN 'peer_review'
      WHEN s.sfx RLIKE '^(g|fig|figure)[-_.]?[0-9]+[a-z]?$' THEN 'figure'
      WHEN s.sfx RLIKE '^(t|tab|table)[-_.]?[0-9]+[a-z]?$' THEN 'table'
      WHEN s.sfx RLIKE '^s[0-9]+$' THEN 'supporting_information'
      ELSE 'supplementary_file' END,
  -- the number in the last segment, when it reads as a count (1-999); an internal file id gets no number
  'num', CASE WHEN try_cast(regexp_extract(s.sfx, '(?<![0-9])([0-9]{1,9})[a-z]?$', 1) AS BIGINT) BETWEEN 1 AND 999
              THEN CAST(try_cast(regexp_extract(s.sfx, '(?<![0-9])([0-9]{1,9})[a-z]?$', 1) AS BIGINT) AS STRING) END,
  -- a real title: not blank, not a file name, not the parent's title again (or nearly: ACS deposits repeat it less a Greek letter)
  'keep_title', coalesce(length(r.rec_title) >= 3
      AND r.prefix <> '10.1021'  -- ACS deposits its supporting files under a draft of the article's title, editors' queries included
      AND NOT lower(r.rec_title) RLIKE '\\.(pdf|docx?|xlsx?|pptx?|csv|tsv|txt|zip|rar|gz|7z|tiff?|jpe?g|png|gif|bmp|eps|svg|mp4|mov|avi|wmv|mpe?g|mp3|wav|cif|cml|fcf|hkl|mol|sdf|pdb|xml|json|rtf|tex|bib|mat|fasta?|nex|tre|odt|ods|html?|py|ipynb|rmd|sav|dta|kml|kmz)$'
      AND NOT (r.rec_title NOT LIKE '% %' AND r.rec_title RLIKE '[_0-9]')
      AND NOT (n.rec = n.par OR (length(n.rec) >= 10 AND n.par LIKE concat('%', n.rec, '%'))
               -- levenshtein with a cap answers -1 past the cap
               OR levenshtein(left(n.rec, 300), left(n.par, 300), 45) BETWEEN 0 AND 0.15 * least(length(n.par), 300)), false),
  -- a bare label ("Supplementary data 2", "Front Matter") says nothing alone: the parent's title is added
  'label_only', coalesce(length(r.rec_title) <= 40
      AND lower(r.rec_title) RLIKE '\\b(supplement\\w*|supporting|additional|appendix|appendices|figures?|fig|tables?|data ?sheets?|datasets?|data|videos?|movies?|audio|images?|files?|matter|presentation|multimedia|online resource)\\b', false)
))) k
WHERE p.type NOT IN ('paratext', 'report', 'dataset') AND length(trim(p.title)) > 0
) x
) s ON t.native_id = s.native_id
WHEN MATCHED AND (t.title <> s.made_title OR NOT (t.license <=> s.parent_license) OR t.is_oa <> s.parent_is_oa
                  OR NOT (t.parent_work_id <=> s.parent_work_id))
THEN UPDATE SET t.title = s.made_title, t.license = s.parent_license, t.is_oa = s.parent_is_oa,
                t.parent_work_id = s.parent_work_id, t.component_kind = s.component_kind, t.title_source = s.title_source

### Checks and tonight's numbers

In [0]:
SELECT
  ASSERT_TRUE(count(*) = count(DISTINCT native_id), 'crossref_component_works: a DOI is listed twice'),
  ASSERT_TRUE(count_if(title IS NULL OR length(trim(title)) = 0) = 0, 'crossref_component_works: an untitled component'),
  ASSERT_TRUE(count_if(parent_doi IS NULL OR parent_work_id IS NULL) = 0, 'crossref_component_works: a component with no parent'),
  ASSERT_TRUE(count_if(native_id LIKE '10.1787/%') = 0, 'crossref_component_works: an OECD component'),
  ASSERT_TRUE(count_if(normalized_title IS NOT NULL OR merge_key.title_author IS NOT NULL) = 0, 'crossref_component_works: a component with a title key')
FROM openalex.crossref.crossref_component_works

In [0]:
SELECT 'component records' AS what, count(*) AS n FROM openalex.crossref.crossref_component_records
UNION ALL SELECT 'admitted components', count(*) FROM openalex.crossref.crossref_component_works
UNION ALL SELECT 'admitted in this run (last 6 h)', count(*) FROM openalex.crossref.crossref_component_works WHERE admitted_at >= current_timestamp() - INTERVAL 6 HOURS
UNION ALL SELECT 'parents', count(DISTINCT parent_work_id) FROM openalex.crossref.crossref_component_works
UNION ALL SELECT concat('kind: ', component_kind), count(*) FROM openalex.crossref.crossref_component_works GROUP BY component_kind
UNION ALL SELECT concat('title: ', title_source), count(*) FROM openalex.crossref.crossref_component_works GROUP BY title_source
UNION ALL SELECT 'admitted, parent no longer a live work', count(*) FROM openalex.crossref.crossref_component_works w
  LEFT ANTI JOIN openalex.works.openalex_works p ON p.id = w.parent_work_id